# 🚀 Modelos de lenguaje abiertos en tu GPU: Qwen, Llama y (opcional) Gemini
*Curso de IA Generativa · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** calcular cuánta memoria pide un modelo, cargar modelos abiertos en la GPU gratis de Colab (completos y
**cuantizados a 4 bits**), medir su velocidad y compararlos con un modelo comercial.

| Modelo | Parámetros | Licencia | Acceso |
|---|---|---|---|
| Qwen2.5-1.5B-Instruct | 1.500 millones | Apache 2.0 | Libre |
| Qwen2.5-7B-Instruct | 7.600 millones | Apache 2.0 | Libre |
| Llama-3.2-3B-Instruct / Llama-3.1-8B-Instruct | 3.200 / 8.000 millones | Licencias Llama 3.2 y 3.1 | Hay que **aceptar la licencia** en su página de Hugging Face y usar tu `HF_TOKEN` |

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os, gc, time
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO, "|", torch.cuda.get_device_name(0) if DISPOSITIVO == "cuda" else "")
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

def liberar():
    """Devuelve la memoria de la GPU después de borrar un modelo («del modelo»)."""
    gc.collect()
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()

In [ ]:
%pip install -q transformers accelerate bitsandbytes

In [ ]:
# Token de Hugging Face (opcional; obligatorio para Llama): panel 🔑 Secretos, nombre HF_TOKEN
if EN_COLAB:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN cargado desde Secretos")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print("Sin HF_TOKEN: los modelos abiertos funcionan igual; Llama no.")

## 1. ¿Cuánta memoria pide un modelo?

Cada parámetro ocupa **4 bytes** en 32 bits, **2** en 16 bits y **medio byte** en 4 bits. A eso se suma la memoria de
la conversación (la caché de atención), que crece con el contexto.

In [ ]:
def memoria_gb(parametros, bits):
    """Memoria aproximada de los pesos (sin la caché de atención)."""
    return parametros * bits / 8 / 1024**3

for nombre, p in [("Qwen2.5-1.5B", 1.5e9), ("Llama-3.2-3B", 3.2e9), ("Qwen2.5-7B", 7.6e9), ("Llama-3.1-8B", 8.0e9)]:
    print(f"{nombre:14} 32 bits: {memoria_gb(p, 32):5.1f} GB · 16 bits: {memoria_gb(p, 16):5.1f} GB · 4 bits: {memoria_gb(p, 4):4.1f} GB")
if DISPOSITIVO == "cuda":
    print(f"\nTu GPU tiene {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Un modelo pequeño, completo en 16 bits

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def cargar(nombre, cuatro_bits=False):
    opciones = {"device_map": "auto"} if DISPOSITIVO == "cuda" else {}
    if cuatro_bits:
        opciones["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                             bnb_4bit_compute_dtype=torch.float16)
    else:
        opciones["torch_dtype"] = torch.float16 if DISPOSITIVO == "cuda" else torch.float32
    if DISPOSITIVO == "cuda":
        torch.cuda.reset_peak_memory_stats()
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, **opciones)
    if DISPOSITIVO == "cuda":
        print(f"{nombre}: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB en la GPU")
    return tok, modelo

def chatear(tok, modelo, pregunta, temperatura=0.0, tokens=200):
    mensajes = [{"role": "user", "content": pregunta}]
    entrada = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_dict=True,
                                      return_tensors="pt").to(modelo.device)
    inicio = time.perf_counter()
    salida = modelo.generate(**entrada, max_new_tokens=tokens, do_sample=temperatura > 0,
                             temperature=temperatura or None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    segundos = time.perf_counter() - inicio
    print(f"[{len(nuevos)} tokens en {segundos:.1f} s · {len(nuevos) / segundos:.1f} tokens/s]")
    return tok.decode(nuevos, skip_special_tokens=True)

PREGUNTA = "Explica en tres frases qué es una GPU y por qué sirve para la inteligencia artificial."
tok, modelo = cargar("Qwen/Qwen2.5-1.5B-Instruct")
print(chatear(tok, modelo, PREGUNTA))

## 3. La temperatura

Con temperatura 0 el modelo elige siempre la palabra más probable: misma pregunta, misma respuesta. Al subirla, se
arriesga más: respuestas más variadas y, a veces, menos precisas.

In [ ]:
for t in (0.0, 0.7, 1.3):
    print(f"── temperatura {t}")
    print(chatear(tok, modelo, "Inventa un nombre para una cafetería de programadores.", temperatura=t, tokens=30), "\n")
del modelo; liberar()

## 4. Un modelo de 7.600 millones de parámetros, cuantizado a 4 bits

En 16 bits no cabría en una GPU de 16 GB con holgura; en **4 bits** (formato NF4 de bitsandbytes) ocupa cerca de 5 GB
y responde casi igual de bien. Requiere GPU.

In [ ]:
if DISPOSITIVO == "cuda":
    tok7, modelo7 = cargar("Qwen/Qwen2.5-7B-Instruct", cuatro_bits=True)
    print(chatear(tok7, modelo7, PREGUNTA))
    del modelo7; liberar()
else:
    print("La cuantización a 4 bits con bitsandbytes necesita GPU.")

## 5. Llama (si aceptaste su licencia)

1. Entra a la página de [Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) con tu cuenta
   de Hugging Face y acepta la licencia (la aprobación puede tardar).
2. Crea un token de lectura en tu cuenta y guárdalo en Secretos como `HF_TOKEN`.
3. Ejecuta la celda: si no tienes acceso, explica por qué y sigue.

In [ ]:
MODELO_LLAMA = "meta-llama/Llama-3.2-3B-Instruct"     # o "meta-llama/Llama-3.1-8B-Instruct" con cuatro_bits=True
try:
    tokl, llama = cargar(MODELO_LLAMA)
    print(chatear(tokl, llama, PREGUNTA))
    del llama; liberar()
except Exception as e:
    print("No se pudo cargar Llama (¿aceptaste la licencia y tienes HF_TOKEN?):", type(e).__name__)

## 6. Un modelo comercial por API: Gemini (opcional)

Google AI Studio da claves de Gemini con un nivel gratuito sujeto a límites; revisa las condiciones vigentes antes de
usarla. La clave va en Secretos como `GEMINI_API_KEY`, nunca en el cuaderno. Revisa también el nombre del modelo
vigente en la documentación de la API.

In [ ]:
MODELO_GEMINI = "gemini-2.5-flash"     # cámbialo por el vigente si este ya no existe
clave = None
if EN_COLAB:
    from google.colab import userdata
    try:
        clave = userdata.get("GEMINI_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print("Sin GEMINI_API_KEY en Secretos: se salta esta parte.")
if clave:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=True)
    from google import genai
    respuesta = genai.Client(api_key=clave).models.generate_content(model=MODELO_GEMINI, contents=PREGUNTA)
    print(respuesta.text)

## Pruébalo tú

Con `memoria_gb`, calcula en `cabe_70b` si los pesos de un modelo de **70.000 millones** de parámetros en 4 bits
caben en una GPU de 16 GB (`True` o `False`).

In [ ]:
cabe_70b = None   # ← tu código

In [ ]:
# Comprobación
if cabe_70b is None:
    print("Aún no lo resuelves.")
elif cabe_70b is False:
    print("✅ Correcto: ~33 GB solo de pesos; no cabe en 16 GB")
else:
    print("❌ Revisa tu respuesta")

## Resumen

- Memoria de los pesos ≈ parámetros × bits ÷ 8. Cuantizar a 4 bits la divide por cuatro respecto de 16 bits.
- En la GPU gratis de Colab caben modelos de hasta unos 8.000 millones de parámetros en 4 bits.
- Llama exige aceptar su licencia; Qwen 1.5B y 7B son Apache 2.0.

**Fuentes:** [Transformers · cuantización con bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes) · [Transformers · plantillas de chat](https://huggingface.co/docs/transformers/chat_templating) · [Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) · [Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) · [Gemini API · SDK google-genai](https://ai.google.dev/gemini-api/docs/quickstart)